# 00. Data preparation - augmented HDF5 datasets

Builds the HDF5 datasets consumed by the nine Keras training notebooks
(01-09) from the Sentinel-2 and Landsat RGB composites of `split_data`. For every original image the file stores the original plus
`NUM_AUGMENTATIONS` augmented copies; images are normalized with the
encoder-specific `preprocess_input` **before** being written, so the training
notebooks read ready-to-use tensors.

Run this notebook six times (or loop over the settings) to produce:

| ENCODER    | SPLIT | Augmented copies | OUTPUT_H5 |
|------------|-------|------------------|-----------|
| `vgg16`    | train | 5 | `train_data_augmented_vgg_norm.h5` |
| `vgg16`    | val   | 0 | `val_data_vgg_norm.h5` |
| `resnet50` | train | 5 | `train_data_augmented_resnet50.h5` |
| `resnet50` | val   | 0 | `val_data_resnet50.h5` |
| `xception` | train | 5 | `train_data_augmented_xception_norm.h5` |
| `xception` | val   | 0 | `val_data_xception_norm.h5` |

Augmentation is applied to the **training split only**; the validation files
contain the original samples without augmentation. The augmentation policy:
horizontal and vertical flips (p = 0.5 each), rotation within +/-25 degrees
(p = 0.9), and adaptive photometric transforms - gamma, linear
contrast/brightness, CLAHE on the L channel and an HSV tweak - whose parameter
ranges depend on the mean brightness of the image (dark / medium / bright
regimes, thresholds 80 and 200). A safeguard keeps the mean brightness of
augmented images above `MIN_MEAN_ALLOWED`. The same policy is applied
on-the-fly by the SegFormer (11) and Mask2Former (12) notebooks; YOLO11-seg
uses its native Ultralytics augmentation.

The test split is **not** converted to HDF5: the training notebooks evaluate
directly on the raw `test/images` + `test/Masks` folders.

In [ ]:
import os
import random

import cv2
import h5py
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

## 1. Configuration

In [ ]:
# ============================================================================
# CONFIGURATION
# ============================================================================

# Encoder whose preprocess_input will be baked into the HDF5 file:
#   "vgg16" | "resnet50" | "xception"
ENCODER = "vgg16"

# Dataset split to convert: "train" | "val"
SPLIT = "train"

# Number of augmented copies per original image.
# Augmentation is applied to the training split only.
NUM_AUGMENTATIONS = 5 if SPLIT == "train" else 0

DATA_ROOT = "split_data"
IMG_DIR = os.path.join(DATA_ROOT, SPLIT, "images")
MASK_DIR = os.path.join(DATA_ROOT, SPLIT, "Masks")

IMG_SIZE = (256, 256)        # (width, height)
MIN_MEAN_ALLOWED = 60.0      # lower bound for the mean brightness of augmented images

_H5_NAMES = {
    "vgg16": "vgg_norm",
    "resnet50": "resnet50",
    "xception": "xception_norm",
}
if SPLIT == "train":
    OUTPUT_H5 = f"train_data_augmented_{_H5_NAMES[ENCODER]}.h5"
else:
    OUTPUT_H5 = f"val_data_{_H5_NAMES[ENCODER]}.h5"

if ENCODER == "vgg16":
    from tensorflow.keras.applications.vgg16 import preprocess_input
elif ENCODER == "resnet50":
    from tensorflow.keras.applications.resnet50 import preprocess_input
elif ENCODER == "xception":
    from tensorflow.keras.applications.xception import preprocess_input
else:
    raise ValueError(f"Unknown encoder: {ENCODER}")

print(f"Encoder: {ENCODER} | Split: {SPLIT} | Augmentations: {NUM_AUGMENTATIONS}")
print(f"Output: {OUTPUT_H5}")

## 2. Image/mask pairing

In [ ]:
def list_image_mask_pairs(img_dir, mask_dir,
                          exts_img=(".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp"),
                          exts_mask=(".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp")):
    """Match images and masks by base name; return (pairs, unmatched lists)."""
    img_files = sorted([f for f in os.listdir(img_dir) if f.lower().endswith(exts_img)])
    mask_files = sorted([f for f in os.listdir(mask_dir) if f.lower().endswith(exts_mask)])
    img_map = {os.path.splitext(f)[0]: f for f in img_files}
    mask_map = {os.path.splitext(f)[0]: f for f in mask_files}
    common = sorted(set(img_map.keys()) & set(mask_map.keys()))
    paired = [(img_map[b], mask_map[b]) for b in common]
    missing_imgs = [f for f in img_files if os.path.splitext(f)[0] not in common]
    missing_masks = [f for f in mask_files if os.path.splitext(f)[0] not in common]
    return paired, missing_imgs, missing_masks

## 3. Adaptive augmentation

In [ ]:
class Augmentor:
    """Shared augmentation policy of the study.

    Identical for every model except YOLO11-seg (which uses the native
    Ultralytics augmentation). Operates on RGB images in [0, 255] (uint8 or
    float32) and binary masks; returns an RGB float32 image in [0, 255] and a
    2D uint8 mask in {0, 1}. Photometric parameter ranges depend on the mean
    brightness of the image so that dark scenes are not darkened further and
    bright scenes are not over-exposed.
    """

    def __init__(self, rotation_range=(-25, 25), rotation_prob=0.9, flip_prob=0.5,
                 min_mean_allowed=60.0):
        self.rotation_range = rotation_range
        self.rotation_prob = rotation_prob
        self.flip_prob = flip_prob
        self.min_mean_allowed = min_mean_allowed

    def augment(self, img_rgb, mask):
        img = np.clip(img_rgb, 0, 255).astype(np.uint8)
        mask_in = (mask > 0.5).astype(np.uint8)

        h, w = img.shape[:2]

        # 1) Geometric augmentations.
        if np.random.rand() < self.flip_prob:
            img = cv2.flip(img, 1)
            mask_in = cv2.flip(mask_in, 1)
        if np.random.rand() < self.flip_prob:
            img = cv2.flip(img, 0)
            mask_in = cv2.flip(mask_in, 0)

        if np.random.rand() < self.rotation_prob:
            angle = float(np.random.uniform(self.rotation_range[0], self.rotation_range[1]))
            Mrot = cv2.getRotationMatrix2D((w / 2.0, h / 2.0), angle, 1.0)
            img = cv2.warpAffine(img, Mrot, (w, h), flags=cv2.INTER_LINEAR,
                                 borderMode=cv2.BORDER_CONSTANT, borderValue=(0, 0, 0))
            mask_in = cv2.warpAffine(mask_in, Mrot, (w, h), flags=cv2.INTER_NEAREST,
                                     borderMode=cv2.BORDER_CONSTANT, borderValue=0)

        # 2) Adaptive photometric transforms driven by the mean brightness.
        gray0 = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY).astype(np.float32)
        mean0 = float(gray0.mean())

        dark_thresh = 80.0
        bright_thresh = 200.0

        if mean0 < dark_thresh:
            # Dark scenes: brighten gently, never darken.
            gamma = float(np.random.uniform(0.8, 1.05))
            alpha = float(np.random.uniform(0.9, 1.15))
            beta = float(np.random.uniform(0, 40))
            apply_clahe = (np.random.rand() < 0.3)
            apply_hsv = (np.random.rand() < 0.25)
        elif mean0 > bright_thresh:
            # Very bright scenes: allow slight darkening.
            gamma = float(np.random.uniform(0.95, 1.25))
            alpha = float(np.random.uniform(0.85, 1.25))
            beta = float(np.random.uniform(-40, 20))
            apply_clahe = (np.random.rand() < 0.5)
            apply_hsv = (np.random.rand() < 0.5)
        else:
            # Mid-tone scenes: moderate transforms.
            gamma = float(np.random.uniform(0.9, 1.1))
            alpha = float(np.random.uniform(0.92, 1.2))
            beta = float(np.random.uniform(-20, 30))
            apply_clahe = (np.random.rand() < 0.45)
            apply_hsv = (np.random.rand() < 0.4)

        # Gamma correction in [0, 1] float space.
        img_f = np.clip(img.astype(np.float32) / 255.0, 0.0, 1.0)
        if abs(gamma - 1.0) > 1e-6:
            img_f = np.power(img_f, gamma)
        img_gc = np.clip(img_f * 255.0, 0, 255).astype(np.float32)

        # Mean-centred linear contrast plus brightness shift.
        gray = cv2.cvtColor(img_gc.astype(np.uint8), cv2.COLOR_RGB2GRAY).astype(np.float32)
        mean_l = gray.mean()
        img_out = (img_gc - mean_l) * alpha + mean_l + beta

        # Moderate CLAHE on the L channel.
        if apply_clahe:
            lab = cv2.cvtColor(np.clip(img_out, 0, 255).astype(np.uint8), cv2.COLOR_RGB2LAB)
            L, A, B = cv2.split(lab)
            clip_limit = np.random.uniform(1.0, 3.0)
            clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=(8, 8))
            L = clahe.apply(L)
            lab = cv2.merge([L, A, B])
            img_out = cv2.cvtColor(lab, cv2.COLOR_LAB2RGB).astype(np.float32)

        # Gentle HSV tweak.
        if apply_hsv:
            hsv = cv2.cvtColor(np.clip(img_out, 0, 255).astype(np.uint8),
                               cv2.COLOR_RGB2HSV).astype(np.float32)
            h_ch, s_ch, v_ch = cv2.split(hsv)
            if mean0 < dark_thresh:
                sat_mul = np.random.uniform(0.9, 1.15)
                val_shift = np.random.uniform(0, 30)
            else:
                sat_mul = np.random.uniform(0.85, 1.25)
                val_shift = np.random.uniform(-30, 30)
            s_ch = np.clip(s_ch * sat_mul, 0, 255)
            v_ch = np.clip(v_ch + val_shift, 0, 255)
            hsv2 = cv2.merge([h_ch, s_ch, v_ch]).astype(np.uint8)
            img_out = cv2.cvtColor(hsv2, cv2.COLOR_HSV2RGB).astype(np.float32)

        # Strict guard against overly dark outputs.
        gray_after = cv2.cvtColor(np.clip(img_out, 0, 255).astype(np.uint8), cv2.COLOR_RGB2GRAY)
        mean_after = float(gray_after.mean())
        if mean_after < self.min_mean_allowed:
            img_out = img_out + (self.min_mean_allowed - mean_after)

        img_final = np.clip(img_out, 0, 255).astype(np.float32)
        mask_final = (mask_in > 0).astype(np.uint8)

        return img_final, mask_final

## 4. HDF5 writer

In [ ]:
def create_h5_with_augmentations(paired_list, img_dir, mask_dir, output_h5,
                                 img_size=IMG_SIZE, num_augs=NUM_AUGMENTATIONS,
                                 augmentor=None, verbose=True):
    """Write original + augmented samples into an HDF5 file.

    HDF5 layout: "images" (N, H, W, 3) float32 with preprocess_input already
    applied, "masks" (N, H, W, 1) float32 in {0, 1}.
    """
    if augmentor is None:
        augmentor = Augmentor(min_mean_allowed=MIN_MEAN_ALLOWED)

    n_pairs = len(paired_list)
    total_samples = n_pairs * (1 + num_augs)

    H, W = img_size[1], img_size[0]  # img_size is (width, height)
    with h5py.File(output_h5, "w") as f:
        d_images = f.create_dataset("images", shape=(total_samples, H, W, 3), dtype=np.float32)
        d_masks = f.create_dataset("masks", shape=(total_samples, H, W, 1), dtype=np.float32)

        idx = 0
        for i, (img_name, mask_name) in enumerate(paired_list):
            img_path = os.path.join(img_dir, img_name)
            mask_path = os.path.join(mask_dir, mask_name)

            bgr = cv2.imread(img_path, cv2.IMREAD_COLOR)
            if bgr is None:
                print(f"Warning: could not load {img_path}, skipping.")
                continue
            img_rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
            img_rgb = cv2.resize(img_rgb, img_size).astype(np.float32)

            mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
            if mask is None:
                print(f"Warning: could not load mask {mask_path}, skipping.")
                continue
            mask = cv2.resize(mask, img_size, interpolation=cv2.INTER_NEAREST)
            mask_bin = (mask > 128).astype(np.uint8)

            # 1) Original sample (encoder-specific normalization).
            img_pre = preprocess_input(img_rgb.copy())
            d_images[idx, ...] = img_pre
            d_masks[idx, ...] = np.expand_dims(mask_bin.astype(np.float32), axis=-1)
            idx += 1

            # 2) Augmented copies.
            for _ in range(num_augs):
                aug_img_rgb, aug_mask = augmentor.augment(img_rgb.copy(), mask_bin.copy())
                aug_pre = preprocess_input(aug_img_rgb.copy())
                d_images[idx, ...] = aug_pre
                d_masks[idx, ...] = np.expand_dims(aug_mask.astype(np.float32), axis=-1)
                idx += 1

            if verbose and (i + 1) % 50 == 0:
                print(f"Processed {i + 1}/{n_pairs} pairs; written samples: {idx}/{total_samples}")

        # Trim the datasets if some pairs were skipped.
        if idx < total_samples:
            print(f"Note: wrote {idx} samples, expected {total_samples}. Trimming datasets.")
            f.create_dataset("images_tmp", data=d_images[:idx])
            f.create_dataset("masks_tmp", data=d_masks[:idx])
            del f["images"]
            del f["masks"]
            f["images"] = f["images_tmp"]
            f["masks"] = f["masks_tmp"]
            del f["images_tmp"]
            del f["masks_tmp"]

    print(f"HDF5 file written: {output_h5}")

## 5. Run

In [ ]:
paired, missing_imgs, missing_masks = list_image_mask_pairs(IMG_DIR, MASK_DIR)
print(f"Paired samples: {len(paired)}; images without mask: {len(missing_imgs)}; "
      f"masks without image: {len(missing_masks)}")
if len(paired) == 0:
    raise RuntimeError("No image-mask pairs found. Check the directories and file extensions.")

create_h5_with_augmentations(paired, IMG_DIR, MASK_DIR, OUTPUT_H5,
                             img_size=IMG_SIZE, num_augs=NUM_AUGMENTATIONS)

with h5py.File(OUTPUT_H5, "r") as f:
    print("images:", f["images"].shape, f["images"].dtype)
    print("masks: ", f["masks"].shape, f["masks"].dtype)